# Step 1: Data Preprocessing and Exploratory Behavioral Analysis

In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: HDDM Data Preprocessing and Exploratory Behavioral Analysis (Step 1)
Description: 
  - Filters and cleans trial-level data for HDDM (Informative Priors).
  - Explicitly standardizes emotion legacy coding ('enj' -> 'rew').
  - Implements Response Coding Audit to ensure data integrity.
  - Generates Fair-Ceiling Diagnostics focusing on Rejection Rates.
  - Prepares the final unconstrained matrix for hierarchical modeling.
=============================================================================
"""

import os
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter, PercentFormatter

# =============================================================================
# GLOBAL CONSTANTS & AESTHETICS (EDA Specific)
# =============================================================================
sns.set_theme(style="ticks", palette="colorblind")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
    'pdf.fonttype': 42
})

ACC_COLOR = "#004D40"
REJ_COLOR = "#900C3F"
ACC_ALPHA = 0.52
REJ_ALPHA = 0.75

# Reaction Time (RT) boundaries in milliseconds
RT_MIN_MS = 150
RT_MAX_MS = 3000

# Behavioral response coding (Original Data)
RESPONSE_ACCEPT = 1
RESPONSE_REJECT = 2
RESPONSE_NONE = 0

# HDDM response coding boundary
HDDM_ACCEPT = 1
HDDM_REJECT = 0

# =============================================================================
# LOGGING SETUP
# =============================================================================
def setup_logger(log_file: str = "step1_data_preparation.log") -> logging.Logger:
    """Initialize logging configuration for process tracking."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(levelname)s: %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# =============================================================================
# MODULE 1: RESPONSE CODING AUDIT
# =============================================================================
def audit_response_coding(df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """Strictly audits the mapping between original button presses and HDDM boundaries."""
    logger.info("--- Executing Response Coding Audit ---")
    
    orig_accept = (df['reaction'] == RESPONSE_ACCEPT).sum()
    orig_reject = (df['reaction'] == RESPONSE_REJECT).sum()
    
    response_mapping = {RESPONSE_ACCEPT: HDDM_ACCEPT, RESPONSE_REJECT: HDDM_REJECT}
    df['response_hddm'] = df['reaction'].map(response_mapping)
    
    hddm_accept = (df['response_hddm'] == HDDM_ACCEPT).sum()
    hddm_reject = (df['response_hddm'] == HDDM_REJECT).sum()
    
    if orig_accept != hddm_accept or orig_reject != hddm_reject:
        raise ValueError("CRITICAL: Response mapping mismatch detected!")
    
    if df['response_hddm'].isnull().any():
        unmapped = df.loc[df['response_hddm'].isnull(), 'reaction'].unique()
        raise ValueError(f"Unmapped response values detected: {unmapped}")
        
    audit_data = [{
        'Original_Response': 'Accept (1)', 'Original_Count': orig_accept,
        'HDDM_Boundary': 'Upper (1)', 'HDDM_Count': hddm_accept
    }, {
        'Original_Response': 'Reject (2)', 'Original_Count': orig_reject,
        'HDDM_Boundary': 'Lower (0)', 'HDDM_Count': hddm_reject
    }]
    
    pd.DataFrame(audit_data).to_csv("response_coding_audit.csv", index=False)
    logger.info("Response coding audit passed and exported to 'response_coding_audit.csv'.")
    return df

# =============================================================================
# MODULE 2: FAIR-CEILING DIAGNOSTICS & ROBUSTNESS CHECKS
# =============================================================================
def generate_fair_ceiling_diagnostics(df_valid: pd.DataFrame, logger: logging.Logger):
    """Calculates rejection rates across Fair and Unfair conditions to justify targeting Unfair trials."""
    logger.info("--- Generating Fair-Ceiling Diagnostics ---")
    
    df_valid = df_valid.copy()
    df_valid['Condition_Type'] = np.where(df_valid['Offers_You'] <= 2, 'Unfair (9:1, 8:2)', 
                                 np.where(df_valid['Offers_You'] >= 4, 'Fair (5:5, 6:4)', 'Intermediate'))
    
    df_target = df_valid[df_valid['Condition_Type'].isin(['Unfair (9:1, 8:2)', 'Fair (5:5, 6:4)'])]
    
    summary = df_target.groupby(['Condition_Type', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()
    
    summary.to_csv("fair_ceiling_diagnostics.csv", index=False)
    logger.info("Fair-ceiling behavior summary exported to 'fair_ceiling_diagnostics.csv'.")
    
    # Simple Visual Verification
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=summary, x='emotion', y='Rejection_Rate', hue='Condition_Type',
        palette=['#4A148C', '#900C3F'], alpha=0.85
    )
    plt.title("Empirical Rejection Rates: Fair vs. Unfair Offers", pad=15)
    plt.ylabel("Probability of Rejection")
    plt.xlabel("Emotion Condition")
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.ylim(0, 1.05)
    plt.legend(title="Offer Type", loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig("fair_unfair_rejection_rate.pdf", dpi=300)
    plt.close()

def generate_offer_ratio_robustness(df_unfair: pd.DataFrame, logger: logging.Logger):
    """Validates whether 9:1 and 8:2 ratios behave similarly enough to merge."""
    logger.info("--- Generating 9:1 vs 8:2 Robustness Check ---")
    
    summary = df_unfair.groupby(['Offers_You', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()
    
    summary['Offer_Ratio'] = summary['Offers_You'].map({1: '9:1', 2: '8:2'})
    summary.to_csv("unfair_offer_ratio_behavior_summary.csv", index=False)
    logger.info("Offer ratio robustness check exported to 'unfair_offer_ratio_behavior_summary.csv'.")

# =============================================================================
# DATA LOADING, FILTERING AND HDDM INGESTION
# =============================================================================
def load_and_filter_data(filepath: str, logger: logging.Logger) -> pd.DataFrame:
    """Loads raw data, applies exclusion criteria, and standardizes labels."""
    logger.info(f"Loading raw data from {filepath}")
    
    df = pd.read_csv(filepath)
    logger.info(f"Initial raw data dimensions: {df.shape}")

    exclusion_log = [{'Stage': 'Total_Initial_Trials', 'Count': len(df)}]

    # 1. Standardize emotion legacy labels EARLY: 'enj' -> 'rew'
    df['emotion'] = df['emotion'].astype(str).str.strip().replace({'enj': 'rew'})

    # 2. Missing responses
    omitted_mask = df['reaction'] == RESPONSE_NONE
    df_valid_resp = df[~omitted_mask]
    exclusion_log.append({'Stage': 'Omitted_Responses', 'Count': omitted_mask.sum()})

    # 3. RT boundaries
    fast_mask = df_valid_resp['RT'] < RT_MIN_MS
    df_valid_rt_low = df_valid_resp[~fast_mask]
    exclusion_log.append({'Stage': 'RT_Too_Fast', 'Count': fast_mask.sum()})

    slow_mask = df_valid_rt_low['RT'] > RT_MAX_MS
    df_valid_rt = df_valid_rt_low[~slow_mask]
    exclusion_log.append({'Stage': 'RT_Too_Slow', 'Count': slow_mask.sum()})

    # Trigger diagnostic before filtering fairness
    generate_fair_ceiling_diagnostics(df_valid_rt, logger)

    # 4. Target Unfair Condition Selection (Offers_You == 1 or 2)
    fair_mask = ~df_valid_rt['Offers_You'].isin([1, 2])
    df_unfair = df_valid_rt[~fair_mask].copy()
    exclusion_log.append({'Stage': 'Non_Unfair_Offers_Excluded', 'Count': fair_mask.sum()})
    exclusion_log.append({'Stage': 'Final_Retained_Unfair_Trials', 'Count': len(df_unfair)})
    
    pd.DataFrame(exclusion_log).to_csv("exclusion_summary_flow.csv", index=False)
    logger.info(f"Retained trials (Unfair conditions only): {len(df_unfair)}")
    
    df_unfair = audit_response_coding(df_unfair, logger)
    generate_offer_ratio_robustness(df_unfair, logger)

    return df_unfair

def prepare_hddm_data(df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """Constructs the canonical data matrix required for HDDM estimation."""
    df = df.copy()
    
    unique_ids = df['participant_id'].unique()
    id_map = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
    df['subj_idx'] = df['participant_id'].map(id_map)
    
    pd.DataFrame(list(id_map.items()), columns=['Original_participant_id', 'HDDM_subj_idx']).to_csv('subject_mapping.csv', index=False)

    hddm_df = pd.DataFrame({
        'subj_idx': df['subj_idx'],
        'rt': df['RT'] / 1000.0,
        'response': df['response_hddm'],
        'emotion': df['emotion'],
        'offer_amount': df['Offers_You']
    })

    return hddm_df

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
def main():
    logger = setup_logger()
    logger.info("="*60)
    logger.info("HDDM DATA PREPARATION PIPELINE INITIATED (Informative Priors)")
    logger.info("="*60)
    
    try:
        input_file = 'trials.csv'
        df_unfair = load_and_filter_data(input_file, logger)
        hddm_df = prepare_hddm_data(df_unfair, logger)
        
        output_file = 'hddm_data_unfair.csv'
        hddm_df.to_csv(output_file, index=False)
        logger.info(f"Final analytical dataset committed to {output_file}")
        
        # Verify emotion categories successfully updated
        emotions_present = hddm_df['emotion'].unique()
        logger.info(f"Emotions preserved for modeling: {emotions_present}")
        
    except Exception as e:
        logger.error(f"Fatal error encountered: {e}")
        raise

if __name__ == "__main__":
    main()

2026-03-23 22:43:18,193 INFO: ============================================================
2026-03-23 22:43:18,194 INFO: HDDM DATA PREPARATION PIPELINE INITIATED (Informative Priors)
2026-03-23 22:43:18,195 INFO: ============================================================
2026-03-23 22:43:18,196 INFO: Loading raw data from trials.csv
2026-03-23 22:43:18,234 INFO: Initial raw data dimensions: (16197, 15)
2026-03-23 22:43:18,249 INFO: --- Generating Fair-Ceiling Diagnostics ---
2026-03-23 22:43:18,273 INFO: Fair-ceiling behavior summary exported to 'fair_ceiling_diagnostics.csv'.
2026-03-23 22:43:18,555 INFO: maxp pruned
2026-03-23 22:43:18,565 INFO: cmap pruned
2026-03-23 22:43:18,568 INFO: kern dropped
2026-03-23 22:43:18,569 INFO: post pruned
2026-03-23 22:43:18,571 INFO: FFTM dropped
2026-03-23 22:43:18,575 INFO: GPOS pruned
2026-03-23 22:43:18,581 INFO: GSUB pruned
2026-03-23 22:43:18,582 INFO: name pruned
2026-03-23 22:43:18,592 INFO: glyf pruned
2026-03-23 22:43:18,594 INFO: Adde

2026-03-23 22:43:18,743 INFO: Closed glyph list over 'glyf': 55 glyphs after
2026-03-23 22:43:18,745 INFO: Glyph names: ['.notdef', '.null', 'C', 'E', 'F', 'O', 'P', 'R', 'T', 'U', 'a', 'b', 'c', 'colon', 'comma', 'd', 'e', 'eight', 'f', 'fi', 'five', 'fl', 'four', 'i', 'j', 'l', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'percent', 'r', 's', 'six', 'space', 't', 'two', 'u', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 'uni239F', 'uni23A0', 'uniFB00', 'uniFB03', 'uniFB04', 'w', 'y', 'zero']
2026-03-23 22:43:18,746 INFO: Glyph IDs:   [0, 1, 2, 3, 8, 11, 12, 15, 19, 20, 21, 23, 24, 25, 27, 28, 29, 38, 40, 41, 50, 51, 53, 55, 56, 68, 69, 70, 71, 72, 73, 76, 77, 79, 80, 81, 82, 83, 85, 86, 87, 88, 90, 92, 3506, 3507, 3508, 3509, 3510, 3511, 5038, 5039, 5040, 5041, 5042]
2026-03-23 22:43:18,747 INFO: Retaining 55 glyphs
2026-03-23 22:43:18,749 INFO: head subsetting not needed
2026-03-23 22:43:18,750 INFO: hhea subsetting not needed
2026-03-23 22:43:18,7

# Step 2a: Global Configuration

In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: HDDM Global Configuration & Lineage Tracking (Step 2a)
Description: 
  - Establishes the Single Source of Truth (SSOT) for the HDDM analytical pipeline.
  - Initializes the CFG object in the Jupyter kernel for direct memory access.
  - Integrates the maximal exploratory model (vazt) to satisfy empirical Occam's razor.
  - Generates a deterministic SHA-256 cryptographic fingerprint of critical 
    structural parameters to enforce downstream data lineage tracking.
  - Serializes a FULLY INSTANTIATED configuration to 'hddm_config.py' 
    and 'config_fingerprint.json', preventing downstream property/method access errors.
=============================================================================
"""

import os
import json
import hashlib
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any

@dataclass
class HDDMConfig:
    """Analytical configuration schema managing MCMC hyperparameters and aesthetics."""
    
    # -------------------------------------------------------------------------
    # 1. OPERATIONAL MODE & DIRECTORY ROUTING
    # -------------------------------------------------------------------------
    run_mode: str = 'final'  
    base_dir: Path = Path(os.getcwd())
    
    # -------------------------------------------------------------------------
    # 2. STRUCTURAL MODELING CONSTRAINTS (EXPERT SPECIFIED)
    # -------------------------------------------------------------------------
    group_only_regressors: bool = True   
    keep_regressor_trace: bool = False   
    p_outlier: float = 0.05
    use_informative_priors: bool = True
    baseline_condition: str = 'neu'
    
    # -------------------------------------------------------------------------
    # 3. EXPERIMENTAL DESIGN & VISUAL STANDARDS (EXCLUDED FROM HASH)
    # -------------------------------------------------------------------------
    emotion_order: List[str] = field(default_factory=lambda: ['neu', 'rew', 'aff', 'dom', 'dis'])
    
    display_labels: Dict[str, str] = field(default_factory=lambda: {
        'neu': 'Neutral', 'rew': 'Reward', 'aff': 'Affiliative',
        'dom': 'Dominance', 'dis': 'Disgust'
    })
    
    colors: Dict[str, str] = field(default_factory=lambda: {
        'neu': "#8491B4", 'rew': "#3C5488", 'aff': "#91D1C2",
        'dom': "#F39B7F", 'dis': "#E64B35"
    })

    # -------------------------------------------------------------------------
    # 4. BASE MCMC HYPERPARAMETERS
    # -------------------------------------------------------------------------
    n_samples: int = 500 if run_mode == 'debug' else 5000
    n_burn: int = 100 if run_mode == 'debug' else 1000
    thin: int = 1
    n_chains: int = 2 if run_mode == 'debug' else 4
    base_seed: int = 2508
    
    # -------------------------------------------------------------------------
    # 5. ADAPTIVE MCMC QUALITY CONTROL (STRICT CONVERGENCE)
    # -------------------------------------------------------------------------
    rhat_cut: float = 1.01   
    rhat_strict: float = 1.01  
    ess_cut: float = 400.0
    ess_strict: float = 400.0  
    mcse_ratio_cut: float = 0.05
    mcse_strict: float = 0.05  
    
    adaptive_batch_size: int = 200 if run_mode == 'debug' else 2500
    max_adaptive_rounds: int = 1 if run_mode == 'debug' else 4

    # -------------------------------------------------------------------------
    # 6. POSTERIOR PREDICTIVE CHECK (PPC) ADEQUACY THRESHOLDS
    # -------------------------------------------------------------------------
    ppc_choice_mae_max: float = 0.05
    ppc_choice_max_err: float = 0.10
    ppc_rt_quantile_mae_max: float = 0.05
    ppc_rt_quantile_max_err: float = 0.10
    ppc_total_samples: int = 50 if run_mode == 'debug' else 500
    
    # -------------------------------------------------------------------------
    # 7. THEORETICAL MODEL ARCHITECTURE DEFINITIONS
    # -------------------------------------------------------------------------
    final_core_models: List[str] = field(default_factory=lambda: ['null', 'v', 'a', 'va'])
    
    # Included the maximal model 'vazt' for rigorous empirical elimination.
    final_exploratory_models: List[str] = field(default_factory=lambda: ['vt', 'vz', 'vat', 'vaz', 'vazt'])
    
    @property
    def final_all_models(self) -> List[str]:
        """Aggregation of core and exploratory architectures."""
        return self.final_core_models + self.final_exploratory_models
    
    model_metadata: Dict[str, Dict[str, Any]] = field(default_factory=lambda: {
        'null': {'tier': 'core', 'varying': []},
        'v':    {'tier': 'core', 'varying': ['v']},
        'a':    {'tier': 'core', 'varying': ['a']},
        'va':   {'tier': 'core', 'varying': ['v', 'a']},
        'vt':   {'tier': 'exploratory', 'varying': ['v', 't']},
        'vz':   {'tier': 'exploratory', 'varying': ['v', 'z']},
        'vat':  {'tier': 'exploratory', 'varying': ['v', 'a', 't']},
        'vaz':  {'tier': 'exploratory', 'varying': ['v', 'a', 'z']},
        'vazt': {'tier': 'exploratory', 'varying': ['v', 'a', 'z', 't']}
    })
    
    model_specs_templates: Dict[str, List[str]] = field(default_factory=lambda: {
        'null': ['v~1', 'a~1', 't~1', 'z~1'],
        'v':    ['v~{ref}', 'a~1', 't~1', 'z~1'],
        'a':    ['v~1', 'a~{ref}', 't~1', 'z~1'],
        'va':   ['v~{ref}', 'a~{ref}', 't~1', 'z~1'],
        'vt':   ['v~{ref}', 'a~1', 't~{ref}', 'z~1'],
        'vz':   ['v~{ref}', 'a~1', 't~1', 'z~{ref}'],
        'vat':  ['v~{ref}', 'a~{ref}', 't~{ref}', 'z~1'],
        'vaz':  ['v~{ref}', 'a~{ref}', 't~1', 'z~{ref}'],
        'vazt': ['v~{ref}', 'a~{ref}', 't~{ref}', 'z~{ref}']
    })

    def initialize_directories(self) -> Dict[str, Path]:
        """Constructs and validates the standard publication output directory tree."""
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery', 
            'figures_main', 'figures_supp', 'tables_main', 'tables_supp'
        ]
        root_out = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {name: root_out / name for name in subdirs}
        for p in paths.values(): 
            p.mkdir(parents=True, exist_ok=True)
        return paths

    def generate_cryptographic_fingerprint(self) -> Dict[str, Any]:
        """
        Extracts structural hyperparameters, applies deterministic sorting, 
        and computes a SHA-256 hash to enforce downstream data lineage.
        Aesthetic parameters are explicitly excluded.
        """
        critical_keys = [
            'run_mode', 'group_only_regressors', 'p_outlier', 'use_informative_priors',
            'baseline_condition', 'n_samples', 'n_burn', 'thin', 'n_chains', 'base_seed',
            'rhat_strict', 'ess_strict', 'final_core_models', 'final_exploratory_models'
        ]
        
        cfg_dict = asdict(self)
        structural_params = {k: cfg_dict[k] for k in critical_keys}
        
        # Deterministic serialization requires sorted keys
        json_str = json.dumps(structural_params, sort_keys=True)
        config_hash = hashlib.sha256(json_str.encode('utf-8')).hexdigest()
        
        return {
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'config_hash': config_hash,
            'structural_parameters': structural_params
        }

# -----------------------------------------------------------------------------
# EXECUTION & PERSISTENCE LOGIC
# -----------------------------------------------------------------------------

# Instance creation for current session memory.
CFG = HDDMConfig()
PATHS = CFG.initialize_directories()

def _serialize_config_to_disk(cfg_obj: HDDMConfig, target_paths: Dict[str, Path]):
    """
    Writes the configuration state to a python module for kernel restarts.
    Crucially, it injects class methods and instantiates the class at the end 
    of the generated file to ensure full object functionality downstream.
    """
    # 1. Python Module Backup (Metaprogramming)
    with open('hddm_config.py', 'w', encoding='utf-8') as f:
        f.write("# -*- coding: utf-8 -*-\n")
        f.write("# Auto-generated configuration recovery module\n")
        f.write("from pathlib import Path\n\n")
        f.write("class _HDDMConfig:\n")
        
        # Write static attributes
        for key, value in cfg_obj.__dict__.items():
            if isinstance(value, Path):
                f.write(f"    {key} = Path(r'{value}')\n")
            elif isinstance(value, str):
                f.write(f"    {key} = '{value}'\n")
            else:
                f.write(f"    {key} = {repr(value)}\n")
        
        # Inject dynamic property
        f.write("\n    @property\n    def final_all_models(self):\n")
        f.write("        return self.final_core_models + self.final_exploratory_models\n")

        # Inject method: initialize_directories
        f.write("\n    def initialize_directories(self):\n")
        f.write("        subdirs = ['manifests', 'audit', 'models', 'ppc', 'recovery', 'figures_main', 'figures_supp', 'tables_main', 'tables_supp']\n")
        f.write("        root_out = self.base_dir / f\"results_hddm_{self.run_mode}\"\n")
        f.write("        paths = {name: root_out / name for name in subdirs}\n")
        f.write("        for p in paths.values(): \n")
        f.write("            p.mkdir(parents=True, exist_ok=True)\n")
        f.write("        return paths\n")
        
        # CRITICAL: Instantiate the singleton
        f.write("\n# Instantiate the configuration to ensure properties and methods are active\n")
        f.write("CFG = _HDDMConfig()\n")

    # 2. Cryptographic Lineage Manifest
    fingerprint_data = cfg_obj.generate_cryptographic_fingerprint()
    manifest_path = target_paths['manifests'] / "config_fingerprint.json"
    
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(fingerprint_data, f, indent=4)
        
    return fingerprint_data['config_hash']

# Execute Serialization
session_hash = _serialize_config_to_disk(CFG, PATHS)

print(f"[{CFG.run_mode.upper()} MODE] Configuration initialized and serialized successfully.")
print(f"Cryptographic Lineage Hash: {session_hash}")
print(f"Convergence Target: R-hat <= {CFG.rhat_cut}, ESS >= {CFG.ess_cut}")

[FINAL MODE] Configuration initialized and serialized successfully.
Cryptographic Lineage Hash: 2a98f2f03f484ce7a2343d487c43a7f06af919e874312b5ca3d293200c227079
Convergence Target: R-hat <= 1.01, ESS >= 400.0


# Step 2b: Hierarchical Bayesian Model Specification and MCMC Estimation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: HDDM Hierarchical Regression Modeling Pipeline (Step 2b - PARALLEL)
Description: 
  - Standardized MCMC estimation constrained to predefined cognitive models.
  - Implements joblib-based multiprocessing for Phase 1 parallel chain execution.
  - Resolves IPC (Inter-Process Communication) overhead by utilizing disk-based 
    model serialization within independent worker processes.
  - Integrates cryptographic data lineage tracking by inheriting the config hash.
=============================================================================
"""

import os
import gc
import json
import time
import logging
import traceback
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import hddm
import arviz as az
import joblib

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & RUNTIME DIRECTORY INITIALIZATION
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

def _initialize_runtime_directories(base_dir: str, run_mode: str) -> Dict[str, Path]:
    """Constructs and validates the standard publication output directory tree."""
    subdirs = [
        'manifests', 'audit', 'models', 'ppc', 'recovery', 
        'figures_main', 'figures_supp', 'tables_main', 'tables_supp'
    ]
    root_out = Path(base_dir) / f"results_hddm_{run_mode}"
    paths = {name: root_out / name for name in subdirs}
    for p in paths.values(): 
        p.mkdir(parents=True, exist_ok=True)
    return paths

PATHS = _initialize_runtime_directories(CFG.base_dir, CFG.run_mode)

# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_estimation_logger() -> logging.Logger:
    """Initializes standardized logging for the MCMC estimation sequence."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_mcmc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    fh = logging.FileHandler(PATHS['audit'] / f'mcmc_estimation_{ts}.log', encoding='utf-8')
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# MULTIPROCESSING WORKER FUNCTION (MUST BE TOP-LEVEL FOR PICKLING)
# =============================================================================
def _parallel_sampling_worker(chain_idx: int, model_name: str, data_path: str, model_dir_str: str) -> Dict[str, Any]:
    """
    Isolated worker process for executing an independent MCMC chain.
    Constructs, samples, and serializes the model to disk to prevent IPC memory bottlenecks.
    """
    # 1. Enforce strict PRNG isolation for this specific process
    chain_seed = CFG.base_seed + (chain_idx * 1000)
    np.random.seed(chain_seed)
    
    # 2. Localized data ingestion and validation
    df = pd.read_csv(data_path)
    df['subj_idx'] = df['subj_idx'].astype(str)
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')
    df = df.dropna(subset=['rt', 'response'])
    
    # 3. Model construction definitions
    ref_str = f'C(emotion, Treatment("{CFG.baseline_condition}"))'
    raw_formulas = CFG.model_specs_templates[model_name]
    formatted_formulas = [f.format(ref=ref_str) for f in raw_formulas]

    base_args = {
        'include': ['v', 'a', 't', 'z'],
        'is_group_model': True,
        'group_only_regressors': getattr(CFG, 'group_only_regressors', True),
        'keep_regressor_trace': getattr(CFG, 'keep_regressor_trace', False),
        'p_outlier': CFG.p_outlier,
        'informative': CFG.use_informative_priors
    }

    model = hddm.HDDMRegressor(df, formatted_formulas, **base_args)
    
    # 4. MAP Initialization
    map_failed = False
    try: 
        model.find_starting_values()
    except Exception: 
        map_failed = True

    # 5. Base MCMC Sampling
    model_dir = Path(model_dir_str)
    db_path = str(model_dir / f"{model_name}_chain_{chain_idx}.db")
    model.sample(CFG.n_samples, burn=CFG.n_burn, thin=CFG.thin, dbname=db_path, db='pickle')
    
    # 6. Disk serialization to avoid returning massive objects over IPC
    hddm_path = str(model_dir / f"{model_name}_chain_{chain_idx}.hddm")
    model.save(hddm_path)
    
    # 7. Safety teardown
    try: 
        model.db.close()
    except Exception: 
        pass

    return {
        'chain_idx': chain_idx,
        'map_failed': map_failed,
        'hddm_path': hddm_path,
        'seed_used': chain_seed
    }

# =============================================================================
# CORE CLASS: MODELING ENGINE
# =============================================================================
class HDDMHierarchicalFitter:
    """Orchestrates structured Bayesian inference with parallel computing and lineage tracking."""
    
    def __init__(self, data_path: str = 'hddm_data_unfair.csv'):
        self.logger = _setup_estimation_logger()
        self.data_path = data_path
        
        self.chain_manifest_records: List[Dict] = []
        self.model_manifest_records: List[Dict] = []
        
        self.config_hash = self._load_and_validate_fingerprint()
        
        self.rhat_cut = getattr(CFG, 'rhat_cut', 1.02)
        self.ess_cut = getattr(CFG, 'ess_cut', 400.0)
        self.mcse_ratio_cut = getattr(CFG, 'mcse_ratio_cut', 0.05)
        self.extra_batch_size = getattr(CFG, 'adaptive_batch_size', 2000 if CFG.run_mode != 'debug' else 200)
        self.max_extra_rounds = getattr(CFG, 'max_adaptive_rounds', 4 if CFG.run_mode != 'debug' else 1)
        
        self.logger.info("=" * 80)
        self.logger.info(f"HDDM ADAPTIVE PARALLEL FITTER INITIATED (Mode: {CFG.run_mode.upper()})")
        self.logger.info(f"Active Cryptographic Lineage Hash: {self.config_hash}")
        
        if CFG.run_mode == 'debug':
            self.logger.warning("!!! RUNNING IN DEBUG MODE: NOT publication-grade. !!!")
            time.sleep(2)  
            
        self.logger.info(f"MCMC Protocol: Samples={CFG.n_samples}, Burn={CFG.n_burn}, Chains={CFG.n_chains}")
        self.logger.info(f"Hardware Allocation: Parallel Execution utilizing {CFG.n_chains} concurrent cores.")
        self.logger.info("=" * 80)

    def _load_and_validate_fingerprint(self) -> str:
        """Enforces data lineage by requiring a valid cryptographic manifest from Step 2a."""
        manifest_path = PATHS['manifests'] / "config_fingerprint.json"
        if not manifest_path.exists():
            raise FileNotFoundError("Lineage manifest missing. Execute Step 2a first.")
        with open(manifest_path, 'r', encoding='utf-8') as f:
            return json.load(f).get('config_hash', 'UNKNOWN_HASH')

    def validate_data(self) -> pd.DataFrame:
        """Loads and enforces datatype constraints for the design matrix."""
        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"Input data {self.data_path} not found.")
        df = pd.read_csv(self.data_path)
        if CFG.baseline_condition not in df['emotion'].unique():
            raise ValueError(f"Baseline condition '{CFG.baseline_condition}' missing.")
        return df

    def _evaluate_posterior_quality(self, models: List[hddm.HDDMRegressor]) -> Tuple[bool, Dict[str, float]]:
        """Extracts mid-flight traces to comprehensively evaluate multi-chain convergence."""
        traces = [m.get_traces() for m in models]
        valid_cols = [c for c in traces[0].columns if not c.startswith(('wfpt', 'mc_', '__'))]
        
        posterior_dict = {col: np.stack([t[col].values for t in traces]) for col in valid_cols}
        idata = az.from_dict(posterior=posterior_dict)
        summary = az.summary(idata, round_to=4)
        
        max_rhat = float(summary['r_hat'].max())
        min_ess_bulk = float(summary['ess_bulk'].min())
        min_ess_tail = float(summary['ess_tail'].min())
        max_mcse_ratio = float((summary['mcse_mean'] / summary['sd'].replace(0, np.nan)).max()) if 'mcse_mean' in summary.columns else 0.0
            
        ok_rhat = max_rhat <= self.rhat_cut
        ok_ess = (min_ess_bulk >= self.ess_cut) and (min_ess_tail >= self.ess_cut)
        ok_mcse = max_mcse_ratio <= self.mcse_ratio_cut
        
        metrics = {
            'max_rhat': max_rhat, 'min_ess_bulk': min_ess_bulk,
            'min_ess_tail': min_ess_tail, 'max_mcse_ratio': max_mcse_ratio
        }
        
        del idata, summary, traces, posterior_dict
        gc.collect()
        
        return (ok_rhat and ok_ess and ok_mcse), metrics

    def fit_single_model(self, model_name: str, df: pd.DataFrame) -> bool:
        """Executes Parallel MCMC base sampling followed by sequential adaptive continuation."""
        model_dir = PATHS['models'] / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        
        metadata = CFG.model_metadata[model_name]
        self.logger.info(f"\nConstructing [{model_name}] (Tier: {metadata['tier'].upper()})")
        
        start_time = time.time()
        map_failed_flags = []
        
        # Phase 1: PARALLEL Base Initialization and Initial Sampling
        self.logger.info(f"  -> Dispatching {CFG.n_chains} independent MCMC chains in PARALLEL...")
        
        worker_results = joblib.Parallel(n_jobs=CFG.n_chains, backend='loky')(
            joblib.delayed(_parallel_sampling_worker)(
                chain_idx, model_name, self.data_path, str(model_dir)
            ) for chain_idx in range(CFG.n_chains)
        )
        
        # Load the completed models back into the main process RAM
        models = []
        for res in sorted(worker_results, key=lambda x: x['chain_idx']):
            self.logger.info(f"     [Parallel Success] Chain {res['chain_idx']+1} ingested. Seed used: {res['seed_used']}")
            map_failed_flags.append(res['map_failed'])
            models.append(hddm.load(res['hddm_path']))
            
        # Phase 2: ArviZ-based Adaptive MCMC Loop (SEQUENTIAL for Database Safety)
        current_samples = CFG.n_samples
        rounds = 0
        final_metrics = {'max_rhat': np.nan, 'min_ess_bulk': np.nan, 'min_ess_tail': np.nan, 'max_mcse_ratio': np.nan}
        
        if CFG.n_chains > 1:
            while rounds < self.max_extra_rounds:
                try:
                    is_converged, metrics = self._evaluate_posterior_quality(models)
                    final_metrics = metrics
                except Exception as e:
                    self.logger.warning(f"     [Warning] Posterior quality check failed: {e}. Terminating adaptive phase.")
                    break

                if is_converged:
                    self.logger.info(f"     [Adaptive] Strict convergence achieved at Round {rounds}.")
                    break
                    
                self.logger.info(f"     [Adaptive] Suboptimal (R-hat: {metrics['max_rhat']:.3f}). Sequential Resampling +{self.extra_batch_size} sweeps...")
                # Adaptive sampling is strictly sequential to avoid corrupting shared DBs
                for chain_idx, model in enumerate(models):
                    db_path = str(model_dir / f"{model_name}_chain_{chain_idx}.db")
                    model.sample(self.extra_batch_size, burn=0, thin=CFG.thin, dbname=db_path, db='pickle')
                    
                current_samples += self.extra_batch_size
                rounds += 1
                
            if rounds >= self.max_extra_rounds and not is_converged:
                self.logger.warning(f"     [Adaptive] Hard limit reached. Terminated with Max R-hat: {final_metrics['max_rhat']:.3f}.")

        # Phase 3: Final Serialization and Metadata Logging
        successful_chains = 0
        for chain_idx, model in enumerate(models):
            dic_score = np.nan
            chain_status = "Failed"
            try:
                dic_score = float(model.dic)
                model.save(str(model_dir / f"{model_name}_chain_{chain_idx}.hddm"))
                successful_chains += 1
                chain_status = "Success"
            except Exception as e:
                self.logger.error(f"     [Fatal] Final serialization failure in chain {chain_idx}: {e}")
            finally:
                self.chain_manifest_records.append({
                    "config_hash": self.config_hash,
                    "model_name": model_name,
                    "chain_id": chain_idx,
                    "random_seed": CFG.base_seed + (chain_idx * 1000),
                    "map_failed": map_failed_flags[chain_idx],
                    "total_samples": current_samples,
                    "dic": dic_score,
                    "status": chain_status
                })
                if hasattr(model, 'db') and hasattr(model.db, 'close'):
                    try: model.db.close()
                    except: pass

        del models
        gc.collect()

        elapsed_mins = (time.time() - start_time) / 60
        self.logger.info(f"  -> Model {model_name} processing concluded in {elapsed_mins:.1f} mins. Final DIC: {dic_score:.1f}")

        self.model_manifest_records.append({
            "config_hash": self.config_hash,
            "model_name": model_name,
            "tier": metadata['tier'],
            "n_chains_completed": successful_chains,
            "total_samples_per_chain": current_samples,
            "adaptive_rounds_used": rounds,
            "final_max_rhat": final_metrics.get('max_rhat', np.nan),
            "final_min_ess_bulk": final_metrics.get('min_ess_bulk', np.nan)
        })

        return successful_chains == CFG.n_chains

    def export_manifests(self):
        """Serializes execution logs to CSV for downstream auditing."""
        pd.DataFrame(self.chain_manifest_records).to_csv(PATHS['manifests'] / "chain_manifest.csv", index=False)
        pd.DataFrame(self.model_manifest_records).to_csv(PATHS['manifests'] / "model_manifest.csv", index=False)
        self.logger.info(f"\nExecution manifests successfully exported to {PATHS['manifests'].name}/ directory.")

    def run(self):
        """Orchestrates the complete estimation pipeline."""
        df = self.validate_data()
        for idx, model_name in enumerate(CFG.final_all_models, 1):
            tier = CFG.model_metadata[model_name]['tier'].upper()
            self.logger.info(f"\n{'='*50}\nProcessing Model {idx}/{len(CFG.final_all_models)}: [{model_name}] ({tier})\n{'='*50}")
            self.fit_single_model(model_name, df)
            
        self.export_manifests()
        self.logger.info("\nALL PARALLEL MCMC ESTIMATION TASKS CONCLUDED.")

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    try:
        fitter = HDDMHierarchicalFitter()
        fitter.run()
    except Exception as e:
        print(f"\nCRITICAL PIPELINE FAILURE: {e}")

23:02:16 - INFO - ================================================================================
23:02:16 - INFO - HDDM ADAPTIVE PARALLEL FITTER INITIATED (Mode: FINAL)
23:02:16 - INFO - Active Cryptographic Lineage Hash: 2a98f2f03f484ce7a2343d487c43a7f06af919e874312b5ca3d293200c227079
23:02:16 - INFO - MCMC Protocol: Samples=5000, Burn=1000, Chains=4
23:02:16 - INFO - Hardware Allocation: Parallel Execution utilizing 4 concurrent cores.
23:02:16 - INFO - ================================================================================
23:02:16 - INFO - 
Processing Model 1/9: [null] (CORE)
23:02:16 - INFO - 
Constructing [null] (Tier: CORE)
23:02:16 - INFO -   -> Dispatching 4 independent MCMC chains in PARALLEL...
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encounter

# Step 3: Posterior Trace Extraction and Predictive Simulation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: Posterior Trace Extraction and Predictive Simulation (Step 3)
Description: 
  - Iteratively processes multi-chain HDDM outputs into ArviZ InferenceData.
  - Enforces strict cryptographic lineage validation against Step 2b artifacts.
  - Generates structural parameter manifests for downstream diagnostics.
  - Executes Posterior Predictive Checks (PPC) and computes response metrics.
  - Injects configuration hash into NetCDF attributes and exported CSVs.
=============================================================================
"""

import os
import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import arviz as az
import hddm

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG
except ImportError:
    raise ImportError("CRITICAL ERROR: 'hddm_config.py' not found. Execute Step 2a first.")

PATHS = CFG.initialize_directories()
PPC_SAMPLES_PER_CHAIN = max(1, CFG.ppc_total_samples // CFG.n_chains)

# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_extraction_logger() -> logging.Logger:
    """Initializes standardized logging for the MCMC extraction and PPC process."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_extraction_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    fh = logging.FileHandler(PATHS['audit'] / f'posterior_extraction_{ts}.log', encoding='utf-8')
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# CORE CLASS: EXTRACTION & PPC ENGINE
# =============================================================================
class HDDMArvizConverter:
    """Handles multi-chain extraction, NetCDF compilation, and lineage tracking."""

    def __init__(self):
        self.logger = _setup_extraction_logger()
        self.observed_stats: List[Dict] = []
        self.ppc_stats: List[Dict] = []
        
        self.active_hash = self._validate_data_lineage()
        
        self.logger.info("=" * 80)
        self.logger.info(f"POSTERIOR EXTRACTION & PPC SIMULATION INITIATED (Mode: {CFG.run_mode.upper()})")
        self.logger.info(f"Active Cryptographic Lineage Hash: {self.active_hash}")
        self.logger.info(f"PPC Protocol: {CFG.ppc_total_samples} total samples ({PPC_SAMPLES_PER_CHAIN}/chain)")
        self.logger.info("=" * 80)

    def _validate_data_lineage(self) -> str:
        """
        Gatekeeper: Ensures the currently loaded CFG mathematically matches the 
        artifacts produced by Step 2b. Prevents stale cache corruption.
        """
        fingerprint_path = PATHS['manifests'] / "config_fingerprint.json"
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        
        if not fingerprint_path.exists() or not manifest_path.exists():
            raise FileNotFoundError("Missing lineage manifests. Execute Step 2a and 2b sequentially.")
            
        with open(fingerprint_path, 'r', encoding='utf-8') as f:
            active_hash = json.load(f).get('config_hash', '')
            
        df_manifest = pd.read_csv(manifest_path)
        if df_manifest.empty:
            raise RuntimeError("Model manifest is empty. Step 2b likely failed.")
            
        # Verify if the artifacts in the manifest match the current fingerprint
        artifact_hash = df_manifest['config_hash'].iloc[0]
        
        if active_hash != artifact_hash:
            raise RuntimeError(
                f"\nCRITICAL LINEAGE MISMATCH DETECTED!\n"
                f"Current Configuration Hash: {active_hash}\n"
                f"Artifact Configuration Hash: {artifact_hash}\n"
                f"Your hddm_config.py was modified without re-running Step 2b. "
                f"Pipeline aborted to prevent statistical ghost states."
            )
            
        return active_hash

    def _generate_posterior_manifest(self, model_name: str, trace_columns: List[str]):
        """Categorizes MCMC parameters into a structural manifest for downstream audit."""
        manifest_records = []
        
        for col in trace_columns:
            if col.startswith(('wfpt', 'mc_', '__')):
                continue
                
            family = col.split('_')[0] if '_' in col else col
            if family not in ['v', 'a', 't', 'z']:
                family = 'other'
                
            level = 'subject' if '_subj' in col else ('sd' if col.endswith('_std') or col.endswith('_var') else 'group')
            is_focal = (level == 'group') and ('Intercept' in col or 'Treatment' in col)
            
            manifest_records.append({
                'variable_name': col,
                'family': family,
                'level': level,
                'focal': is_focal
            })
            
        df_manifest = pd.DataFrame(manifest_records)
        df_manifest.to_csv(PATHS['audit'] / f"posterior_manifest_{model_name}.csv", index=False)

    def _compute_long_format_stats(
        self, df: pd.DataFrame, source: str, model_name: str, chain_id: int = 0, draw_id: int = 0
    ) -> List[Dict]:
        """Calculates standardized statistical summaries structured into a tidy long-format dictionary."""
        records = []
        rt_col = 'rt' if source == 'observed' else 'rt_sampled'
        resp_col = 'response' if source == 'observed' else 'response_sampled'
        
        df = df.copy()
        df[rt_col] = np.abs(df[rt_col])
        emotions = df['emotion'].unique()
        
        for emo in emotions:
            subset = df[df['emotion'] == emo]
            if len(subset) == 0: continue
            
            rejection_rate = (subset[resp_col] == 0).mean()
            mean_rt = subset[rt_col].mean()
            median_rt = subset[rt_col].median()
            
            # Inject Lineage Hash into every record
            base_info = {
                'config_hash': self.active_hash,
                'model_name': model_name, 'chain_id': chain_id, 'draw_id': draw_id,
                'emotion': emo, 'response_type': 'all'
            }
            
            records.extend([
                {**base_info, 'stat_name': 'rejection_rate', 'stat_value': rejection_rate},
                {**base_info, 'stat_name': 'rt_mean', 'stat_value': mean_rt},
                {**base_info, 'stat_name': 'rt_median', 'stat_value': median_rt}
            ])
            
            for resp_val, resp_label in zip([1, 0], ['accept', 'reject']):
                sub_resp = subset[subset[resp_col] == resp_val]
                if len(sub_resp) < 5: 
                    continue
                    
                q10, q50, q90 = np.quantile(sub_resp[rt_col], [0.10, 0.50, 0.90])
                resp_info = {**base_info, 'response_type': resp_label}
                records.extend([
                    {**resp_info, 'stat_name': 'rt_q10', 'stat_value': q10},
                    {**resp_info, 'stat_name': 'rt_q50', 'stat_value': q50},
                    {**resp_info, 'stat_name': 'rt_q90', 'stat_value': q90}
                ])
                
        return records

    def process_model_chains(self, model_name: str):
        """Iteratively loads chains, generates PPC distributions, and compiles ArviZ NetCDF."""
        model_dir = PATHS['models'] / model_name
        out_nc_file = PATHS['models'] / f"{model_name}_arviz.nc"
        
        if out_nc_file.exists():
            # In a strict pipeline, we might force recompilation or read the existing NC's hash
            self.logger.info(f"Skipping [{model_name}]: NetCDF compilation already exists.")
            return

        self.logger.info(f"\nProcessing Extraction for [{model_name}]")
        
        posterior_arrays: Dict[str, List[np.ndarray]] = {}
        ppc_rt_list: List[np.ndarray] = []
        ppc_resp_list: List[np.ndarray] = []
        observed_dict: Optional[Dict[str, np.ndarray]] = None
        
        for chain_idx in range(CFG.n_chains):
            model_file = model_dir / f"{model_name}_chain_{chain_idx}.hddm"
            
            if not model_file.exists():
                self.logger.warning(f"  [Warning] Missing chain artifact: {model_file.name}.")
                continue

            self.logger.info(f"  -> Ingesting Chain {chain_idx + 1}/{CFG.n_chains} into RAM...")
            
            try:
                model = hddm.load(str(model_file))
                traces_df = model.get_traces()
                
                if chain_idx == 0:
                    valid_params = [c for c in traces_df.columns if not c.startswith(('wfpt', 'mc_', '__'))]
                    posterior_arrays = {p: [] for p in valid_params}
                    self._generate_posterior_manifest(model_name, valid_params)
                
                for param in valid_params:
                    posterior_arrays[param].append(traces_df[param].values.astype(np.float32))

                self.logger.info(f"     Simulating PPC distributions (Samples: {PPC_SAMPLES_PER_CHAIN})...")
                ppc_df = hddm.utils.post_pred_gen(model, samples=PPC_SAMPLES_PER_CHAIN, append_data=True)
                
                if observed_dict is None:
                    observed_dict = {
                        'rt': ppc_df['rt'].values.astype(np.float32),
                        'response': ppc_df['response'].values.astype(np.int8),
                        'emotion': ppc_df['emotion'].values.astype(str)
                    }
                    empirical_df = pd.DataFrame(observed_dict)
                    self.observed_stats.extend(
                        self._compute_long_format_stats(empirical_df, 'observed', model_name)
                    )

                n_trials = len(ppc_df)
                rt_sim = np.zeros((n_trials, PPC_SAMPLES_PER_CHAIN), dtype=np.float32)
                resp_sim = np.zeros((n_trials, PPC_SAMPLES_PER_CHAIN), dtype=np.int8)
                
                for i in range(n_trials):
                    rt_sim[i, :] = ppc_df.iloc[i]['rt_sampled']
                    resp_sim[i, :] = np.round(ppc_df.iloc[i]['response_sampled']).astype(np.int8)
                
                ppc_rt_list.append(rt_sim.T)
                ppc_resp_list.append(resp_sim.T)
                
                for draw_offset in range(PPC_SAMPLES_PER_CHAIN):
                    draw_df = ppc_df[['emotion']].copy()
                    draw_df['rt_sampled'] = rt_sim[:, draw_offset]
                    draw_df['response_sampled'] = resp_sim[:, draw_offset]
                    
                    self.ppc_stats.extend(
                        self._compute_long_format_stats(draw_df, 'simulated', model_name, chain_idx, draw_offset)
                    )

            except Exception as e:
                self.logger.error(f"     [FATAL] Failure parsing chain {chain_idx}: {e}")
                continue
                
            finally:
                if 'model' in locals():
                    if hasattr(model, 'db') and hasattr(model.db, 'close'):
                        try: model.db.close()
                        except: pass
                    del model
                if 'ppc_df' in locals(): del ppc_df
                if 'traces_df' in locals(): del traces_df
                gc.collect()

        if posterior_arrays and ppc_rt_list:
            self.logger.info("  Aggregating arrays and constructing NetCDF...")
            try:
                final_posterior = {p: np.stack(c_list, axis=0) for p, c_list in posterior_arrays.items()}
                final_ppc = {
                    'rt': np.stack(ppc_rt_list, axis=0),
                    'response': np.stack(ppc_resp_list, axis=0)
                }
                
                idata = az.from_dict(
                    posterior=final_posterior,
                    posterior_predictive=final_ppc,
                    observed_data=observed_dict
                )
                
                # INJECT LINEAGE HASH INTO NETCDF ATTRIBUTES
                idata.posterior.attrs['config_hash'] = self.active_hash
                idata.posterior.attrs['model_identifier'] = model_name
                idata.posterior.attrs['compilation_timestamp'] = datetime.utcnow().isoformat() + 'Z'
                
                idata.to_netcdf(str(out_nc_file))
                file_mb = out_nc_file.stat().st_size / (1024 * 1024)
                self.logger.info(f"  [SUCCESS] Compiled {out_nc_file.name} ({file_mb:.1f} MB)")
                
            except Exception as e:
                self.logger.error(f"  [FATAL] Error during ArviZ compilation: {e}")

    def export_long_format_summaries(self):
        """Exports the aggregated Tidy dataframes with lineage stamps for downstream visualization."""
        if self.observed_stats:
            df_obs = pd.DataFrame(self.observed_stats).drop_duplicates()
            df_obs.to_csv(PATHS['ppc'] / "observed_summary_long.csv", index=False)
            
        if self.ppc_stats:
            df_ppc = pd.DataFrame(self.ppc_stats)
            df_ppc.to_csv(PATHS['ppc'] / "ppc_summary_long.csv", index=False)
            
        self.logger.info(f"\nLong-format PPC summaries exported to {PATHS['ppc'].name}/ directory.")

    def run(self):
        """Orchestrates the sequence across the defined model space."""
        for model_name in CFG.final_all_models:
            self.process_model_chains(model_name)
            
        self.export_long_format_summaries()
        self.logger.info("\nALL EXTRACTION AND SIMULATION TASKS COMPLETED SUCCESSFULLY")

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    converter = HDDMArvizConverter()
    converter.run()


[CONFIG WARNING] Operating in DEBUG mode. MCMC severely reduced.



15:46:36 - INFO - ======================================================================
15:46:36 - INFO - HDDM MULTI-CHAIN EXTRACTION & ARVIZ COMPILATION (STANDARD TIER)
15:46:36 - INFO - Target Hierarchy: FULL
15:46:36 - INFO - Active Chains:    2
15:46:36 - INFO - PPC Protocol:     50 total samples (25/chain)
15:46:36 - INFO - ======================================================================
15:46:36 - INFO - 
Processing Compilation for [null] -> null_model_inf_full
15:46:36 - INFO -   -> Ingesting Chain 1/2 into RAM...
15:46:38 - INFO -      Identified 128 distinct inference parameters.
15:46:38 - INFO -      Simulating PPC distributions (Samples: 25)...


Start generating posterior prediction...


15:47:25 - INFO -   -> Ingesting Chain 2/2 into RAM...
15:47:27 - INFO -      Simulating PPC distributions (Samples: 25)...


Start generating posterior prediction...


...

15:55:54 - INFO -   Aggregating dimensional arrays and constructing NetCDF...
15:55:54 - INFO -   Serializing InferenceData object...
15:55:57 - INFO -   [SUCCESS] Compiled vz_emotion_arviz.nc (38.2 MB)
15:55:57 - INFO - 
Processing Compilation for [vt] -> vt_emotion_inf_full
15:55:57 - INFO -   -> Ingesting Chain 1/2 into RAM...
15:56:01 - INFO -      Identified 136 distinct inference parameters.
15:56:01 - INFO -      Simulating PPC distributions (Samples: 25)...


Start generating posterior prediction...


15:56:56 - INFO -   -> Ingesting Chain 2/2 into RAM...
15:56:59 - INFO -      Simulating PPC distributions (Samples: 25)...


Start generating posterior prediction...


15:57:55 - INFO -   Aggregating dimensional arrays and constructing NetCDF...
15:57:55 - INFO -   Serializing InferenceData object...
15:57:58 - INFO -   [SUCCESS] Compiled vt_emotion_arviz.nc (39.6 MB)
15:57:58 - INFO - 
15:57:58 - INFO - ALL EXTRACTION AND SIMULATION TASKS COMPLETED SUCCESSFULLY
15:57:58 - INFO - ======================================================================


# Step 4: Convergence Diagnostics and Model Selection

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: Convergence Diagnostics and Model Selection (Step 4)
Description: 
  - Implements a rigorous Four-Level Evaluation Funnel (Technical, 
    Convergence, PPC Adequacy, Relative Comparison).
  - Enforces cryptographic data lineage by validating upstream artifact hashes.
  - Isolates Focal Parameters (group-level effects) for targeted auditing.
  - Automatically selects the optimal model and generates publication-ready 
    diagnostic visualizations, stamping the final audit with the config hash.
=============================================================================
"""

import os
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG
except ImportError:
    raise ImportError("CRITICAL ERROR: 'hddm_config.py' not found. Execute Step 2a first.")

PATHS = CFG.initialize_directories()

# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_diagnostic_logger() -> logging.Logger:
    """Initializes standardized logging for the diagnostic funnel."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_diagnostics_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    fh = logging.FileHandler(PATHS['audit'] / f'model_selection_funnel_{ts}.log', encoding='utf-8')
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# CORE CLASS: DIAGNOSTIC FUNNEL ENGINE
# =============================================================================
class DiagnosticFunnelEngine:
    """Executes hierarchical model evaluation, selects optimal architecture, and tracks lineage."""

    def __init__(self):
        self.logger = _setup_diagnostic_logger()
        self.comparison_records: List[Dict] = []
        
        self.obs_ppc_path = PATHS['ppc'] / "observed_summary_long.csv"
        self.sim_ppc_path = PATHS['ppc'] / "ppc_summary_long.csv"
        
        # Lineage Tracking Validation
        self.active_hash = self._validate_data_lineage()
        
        self.logger.info("=" * 80)
        self.logger.info(f"FOUR-LEVEL DIAGNOSTIC FUNNEL INITIATED (Mode: {CFG.run_mode.upper()})")
        self.logger.info(f"Active Cryptographic Lineage Hash: {self.active_hash}")
        self.logger.info(f"Strict Convergence Criteria: R-hat <= {CFG.rhat_strict}, ESS >= {CFG.ess_strict}")
        self.logger.info(f"PPC Adequacy Criteria:       MAE_Choice <= {CFG.ppc_choice_mae_max}, MAE_RT <= {CFG.ppc_rt_quantile_mae_max}")
        self.logger.info("=" * 80)

    def _validate_data_lineage(self) -> str:
        """
        Gatekeeper: Verifies that the configuration hash matches the downstream 
        artifacts generated by Step 3, preventing evaluation of stale/ghost data.
        """
        fingerprint_path = PATHS['manifests'] / "config_fingerprint.json"
        if not fingerprint_path.exists():
            raise FileNotFoundError("Missing lineage fingerprint. Execute Step 2a first.")
            
        with open(fingerprint_path, 'r', encoding='utf-8') as f:
            active_hash = json.load(f).get('config_hash', '')
            
        if not self.sim_ppc_path.exists():
            raise FileNotFoundError("PPC summary missing. Execute Step 3 first.")
            
        # Check a snippet of the PPC data for the stamped hash
        df_ppc_snippet = pd.read_csv(self.sim_ppc_path, nrows=1)
        if 'config_hash' not in df_ppc_snippet.columns:
            raise RuntimeError("Legacy PPC data detected without config_hash. Please re-run Step 3.")
            
        artifact_hash = df_ppc_snippet['config_hash'].iloc[0]
        
        if active_hash != artifact_hash:
            raise RuntimeError(
                f"\nCRITICAL LINEAGE MISMATCH DETECTED!\n"
                f"Current Configuration Hash: {active_hash}\n"
                f"Artifact Configuration Hash: {artifact_hash}\n"
                f"Your configuration changed without re-running MCMC & Extraction. Pipeline aborted."
            )
            
        return active_hash

    def _get_focal_parameters(self, model_name: str) -> List[str]:
        """Retrieves group-level intercept and treatment parameters from the manifest."""
        manifest_path = PATHS['audit'] / f"posterior_manifest_{model_name}.csv"
        if not manifest_path.exists():
            return []
        df = pd.read_csv(manifest_path)
        return df[df['focal'] == True]['variable_name'].tolist()

    def evaluate_level1_technical(self, model_name: str) -> bool:
        """Verifies the existence of compiled ArviZ NetCDF artifacts."""
        nc_file = PATHS['models'] / f"{model_name}_arviz.nc"
        return nc_file.exists()

    def evaluate_level2_convergence(self, idata: az.InferenceData, model_name: str, focal_params: List[str]) -> Dict[str, Any]:
        """Calculates strict MCMC health metrics exclusively for focal parameters."""
        if not focal_params:
            return {'pass': False, 'max_rhat': np.nan, 'min_ess_bulk': np.nan, 'max_mcse_ratio': np.nan}

        summary_df = az.summary(idata, var_names=focal_params, round_to=4)
        summary_df['mcse_mean_ratio'] = summary_df['mcse_mean'] / summary_df['sd'].replace(0, np.nan)
        
        summary_df.to_csv(PATHS['audit'] / f"focal_parameter_audit_{model_name}.csv")
        
        max_rhat = float(summary_df['r_hat'].max())
        min_ess_bulk = float(summary_df['ess_bulk'].min())
        max_mcse_ratio = float(summary_df['mcse_mean_ratio'].max())
        
        is_converged = (
            (max_rhat <= CFG.rhat_strict) and 
            (min_ess_bulk >= CFG.ess_strict) and 
            (max_mcse_ratio <= CFG.mcse_strict)
        )
        
        return {
            'pass': is_converged,
            'max_rhat': max_rhat,
            'min_ess_bulk': min_ess_bulk,
            'max_mcse_ratio': max_mcse_ratio
        }

    def evaluate_level3_ppc(self, model_name: str) -> Dict[str, Any]:
        """Quantifies Absolute Error (MAE) between empirical behavior and posterior predictions."""
        if not self.obs_ppc_path.exists() or not self.sim_ppc_path.exists():
            return {'pass': False, 'choice_mae': np.nan, 'rt_mae': np.nan}
            
        df_obs = pd.read_csv(self.obs_ppc_path)
        df_sim = pd.read_csv(self.sim_ppc_path)
        
        df_sim_model = df_sim[df_sim['model_name'] == model_name]
        if df_sim_model.empty:
            return {'pass': False, 'choice_mae': np.nan, 'rt_mae': np.nan}
            
        sim_agg = df_sim_model.groupby(['emotion', 'response_type', 'stat_name'])['stat_value'].mean().reset_index()
        sim_agg = sim_agg.rename(columns={'stat_value': 'sim_value'})
        
        merged = pd.merge(df_obs, sim_agg, on=['emotion', 'response_type', 'stat_name'], how='inner')
        merged['abs_error'] = np.abs(merged['stat_value'] - merged['sim_value'])
        
        choice_errs = merged[merged['stat_name'] == 'rejection_rate']['abs_error']
        rt_errs = merged[merged['stat_name'].str.startswith('rt_')]['abs_error']
        
        choice_mae = choice_errs.mean() if not choice_errs.empty else np.nan
        rt_mae = rt_errs.mean() if not rt_errs.empty else np.nan
        choice_max = choice_errs.max() if not choice_errs.empty else np.nan
        rt_max = rt_errs.max() if not rt_errs.empty else np.nan
        
        is_ppc_adequate = (
            (choice_mae <= CFG.ppc_choice_mae_max) and (choice_max <= CFG.ppc_choice_max_err) and
            (rt_mae <= CFG.ppc_rt_quantile_mae_max) and (rt_max <= CFG.ppc_rt_quantile_max_err)
        )
        
        return {
            'pass': is_ppc_adequate,
            'choice_mae': choice_mae,
            'rt_mae': rt_mae
        }

    def plot_winner_diagnostics(self, model_name: str, focal_params: List[str]):
        """Generates targeted, publication-ready MCMC diagnostics for the winning model."""
        self.logger.info(f"Generating supplementary diagnostic plots for optimal model: [{model_name}]")
        nc_file = PATHS['models'] / f"{model_name}_arviz.nc"
        idata = az.from_netcdf(str(nc_file))
        
        az.plot_trace(idata, var_names=focal_params, compact=True, figsize=(12, 2.5 * len(focal_params)))
        plt.tight_layout()
        plt.savefig(PATHS['figures_supp'] / f"diagnostics_trace_{model_name}.pdf", dpi=300)
        plt.close()
        
        az.plot_rank(idata, var_names=focal_params, kind='vlines', vlines_kwargs={'lw':0}, marker='v')
        plt.tight_layout()
        plt.savefig(PATHS['figures_supp'] / f"diagnostics_rank_{model_name}.pdf", dpi=300)
        plt.close()
        
        az.plot_ess(idata, var_names=focal_params, kind='local', marker='_', textsize=10)
        plt.tight_layout()
        plt.savefig(PATHS['figures_supp'] / f"diagnostics_ess_{model_name}.pdf", dpi=300)
        plt.close()
        
        del idata

    def execute_funnel(self):
        """Processes all defined models through the evaluation sequence."""
        for model_name in CFG.final_all_models:
            self.logger.info(f"\nEvaluating [{model_name}]...")
            record = {
                'config_hash': self.active_hash,  # Lineage Injection
                'model_name': model_name,
                'tier': CFG.model_metadata[model_name]['tier'],
                'varying': "-".join(CFG.model_metadata[model_name]['varying']) if CFG.model_metadata[model_name]['varying'] else "none",
                'technical_pass': False,
                'convergence_pass': False,
                'ppc_pass': False,
                'eligible': False,
                'dic': np.nan,
                'selection_rationale': 'Failed Technical'
            }
            
            if not self.evaluate_level1_technical(model_name):
                self.logger.warning("  -> Failed Technical Pass (Missing artifacts).")
                self.comparison_records.append(record)
                continue
            record['technical_pass'] = True
            
            nc_file = PATHS['models'] / f"{model_name}_arviz.nc"
            idata = az.from_netcdf(str(nc_file))
            try: record['dic'] = float(idata.posterior.attrs.get('dic_score', np.nan))
            except: pass
            
            focal_params = self._get_focal_parameters(model_name)
            
            # Convergence Check
            conv_stats = self.evaluate_level2_convergence(idata, model_name, focal_params)
            record['convergence_pass'] = conv_stats['pass']
            record.update({'max_rhat': conv_stats['max_rhat'], 'min_ess': conv_stats['min_ess_bulk']})
            self.logger.info(f"  -> Convergence: {conv_stats['pass']} (R-hat: {conv_stats['max_rhat']:.3f}, ESS: {conv_stats['min_ess_bulk']:.1f})")
            
            del idata 
            
            # PPC Check
            ppc_stats = self.evaluate_level3_ppc(model_name)
            record['ppc_pass'] = ppc_stats['pass']
            record.update({'choice_mae': ppc_stats['choice_mae'], 'rt_mae': ppc_stats['rt_mae']})
            self.logger.info(f"  -> PPC Adequacy: {ppc_stats['pass']} (Choice MAE: {ppc_stats['choice_mae']:.3f}, RT MAE: {ppc_stats['rt_mae']:.3f})")
            
            # Eligibility
            if record['convergence_pass'] and record['ppc_pass']:
                record['eligible'] = True
                record['selection_rationale'] = 'Eligible Candidate'
            elif record['convergence_pass']:
                record['selection_rationale'] = 'Failed PPC'
            else:
                record['selection_rationale'] = 'Failed Convergence'
                
            self.comparison_records.append(record)

    def select_optimal_model(self):
        """Level 4: Identifies the winning model based on DIC and exports stamped audits."""
        if not self.comparison_records:
            self.logger.error("No models evaluated.")
            return

        df = pd.DataFrame(self.comparison_records)
        df['is_winner'] = False
        
        eligible_df = df[df['eligible'] == True]
        
        if eligible_df.empty:
            self.logger.error("\nCRITICAL: No models passed both Convergence and PPC criteria.")
            if CFG.run_mode == 'debug':
                self.logger.warning("DEBUG MODE: Forcing selection of lowest DIC model despite failures.")
                winner_idx = df['dic'].idxmin()
                df.at[winner_idx, 'is_winner'] = True
                df.at[winner_idx, 'selection_rationale'] = 'Winner (Forced by Debug Fallback)'
                winning_model = df.at[winner_idx, 'model_name']
            else:
                # Still export the summary so the user knows exactly why everything failed
                df.to_csv(PATHS['tables_main'] / "model_comparison_summary.csv", index=False)
                raise RuntimeError("Funnel collapsed. No eligible models for final inference.")
        else:
            winner_idx = eligible_df['dic'].idxmin()
            df.at[winner_idx, 'is_winner'] = True
            df.at[winner_idx, 'selection_rationale'] = 'Winner (Strict Convergence + PPC + Lowest DIC)'
            winning_model = df.at[winner_idx, 'model_name']

        # Order columns to prioritize Lineage Hash
        columns_order = [
            'config_hash', 'model_name', 'tier', 'varying', 'dic', 'is_winner', 'eligible', 
            'convergence_pass', 'ppc_pass', 'max_rhat', 'min_ess', 'choice_mae', 'rt_mae', 'selection_rationale'
        ]
        df = df[columns_order]
        
        # Export Stamped Main Summary and Audit
        df.to_csv(PATHS['tables_main'] / "model_comparison_summary.csv", index=False)
        df[df['is_winner'] == True].to_csv(PATHS['audit'] / "final_model_selection_audit.csv", index=False)
        
        self.logger.info("\n" + "="*80)
        self.logger.info(f"OPTIMAL MODEL SELECTED: [{winning_model.upper()}]")
        self.logger.info("="*80)
        
        focal_params = self._get_focal_parameters(winning_model)
        if focal_params:
            self.plot_winner_diagnostics(winning_model, focal_params)

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    funnel = DiagnosticFunnelEngine()
    funnel.execute_funnel()
    funnel.select_optimal_model()

16:10:58 - INFO - ================================================================================
16:10:58 - INFO - MCMC CONVERGENCE, QUANTITATIVE PPC & ROBUST MODEL SELECTION
16:10:58 - INFO - Target Hierarchy: FULL
16:10:58 - INFO - Strict Standard:  R-hat <= 1.01, ESS >= 400.0
16:10:58 - INFO - PPC Max MAE:      Choice <= 0.05, RT <= 0.05
16:10:58 - INFO - ================================================================================



[CONFIG WARNING] Operating in DEBUG mode. MCMC severely reduced.



16:10:59 - INFO - 
Evaluating MCMC convergence for [null]...
16:11:01 - INFO -   -> Max R-hat: 1.241 | Min ESS Bulk: 6.7 | Max MCSE Ratio: 0.489
16:11:03 - INFO -   Quantifying PPC Adequacy for [null]...
/opt/conda/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1559: RuntimeWarning: All-NaN slice encountered
  r, k = function_base._ureduce(a,
16:11:04 - INFO -   -> PPC Choice MAE: 0.057 | PPC RT MAE: nan | Adequacy Pass: False
16:11:05 - INFO - 
Evaluating MCMC convergence for [v]...
16:11:06 - INFO -   -> Max R-hat: 1.100 | Min ESS Bulk: 22.5 | Max MCSE Ratio: 0.291
16:11:09 - INFO -   Quantifying PPC Adequacy for [v]...
/opt/conda/lib/python3.9/site-packages/numpy/lib/nanfunctions.py:1559: RuntimeWarning: All-NaN slice encountered
  r, k = function_base._ureduce(a,
16:11:10 - INFO -   -> PPC Choice MAE: 0.059 | PPC RT MAE: nan | Adequacy Pass: False
16:11:11 - INFO - 
Evaluating MCMC convergence for [a]...
16:11:12 - INFO -   -> Max R-hat: 1.312 | Min ESS Bulk: 5.5 | Max MCSE 

RuntimeError: CRITICAL: No models passed the minimum Eligibility threshold. Do not proceed to interpretation.

# Step 5: Statistical Inference and Publication-Ready Visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: Posterior Inference and Final Visualization (Step 5)
Description: 
  - Validates cryptographic data lineage from the selection funnel.
  - Extracts the optimal model's joint posterior distribution.
  - Computes exact Bayesian inferential statistics (95% HDI, Directional P).
  - Generates publication-ready overlapping KDE plots with HDI annotations.
  - Exports strict academic reporting tables.
=============================================================================
"""

import os
import gc
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG
except ImportError:
    raise ImportError("CRITICAL ERROR: 'hddm_config.py' not found. Execute Step 2a first.")

PATHS = CFG.initialize_directories()

# Apply strict academic visualization aesthetics (Nature/APA style)
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False, 
    'axes.spines.right': False,
    'font.size': 12, 
    'pdf.fonttype': 42, 
    'ps.fonttype': 42,
    'axes.labelsize': 14,
    'legend.fontsize': 11,
    'legend.frameon': False
})

# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_inference_logger() -> logging.Logger:
    """Initializes standardized logging for the final statistical inference module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_inference_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    fh = logging.FileHandler(PATHS['audit'] / f'statistical_inference_{ts}.log', encoding='utf-8')
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# CORE CLASS: BAYESIAN INFERENCE ENGINE
# =============================================================================
class BayesianInferenceEngine:
    """Orchestrates posterior extraction, hypothesis testing, and academic plotting."""

    def __init__(self):
        self.logger = _setup_inference_logger()
        self.active_hash: str = ""
        self.winning_model: str = ""
        self.inference_data: az.InferenceData = None
        self.stats_records: List[Dict] = []
        
        self.logger.info("=" * 80)
        self.logger.info(f"POSTERIOR INFERENCE & VISUALIZATION INITIATED (Mode: {CFG.run_mode.upper()})")
        
        self._validate_and_load_winning_model()
        
    def _validate_and_load_winning_model(self):
        """
        Enforces cryptographic lineage. Ensures the optimal model matches the 
        current analytical configuration before proceeding with statistical tests.
        """
        fingerprint_path = PATHS['manifests'] / "config_fingerprint.json"
        audit_path = PATHS['audit'] / "final_model_selection_audit.csv"
        
        if not fingerprint_path.exists() or not audit_path.exists():
            raise FileNotFoundError("Missing lineage or audit manifests. Execute Steps 2a-4 sequentially.")
            
        with open(fingerprint_path, 'r', encoding='utf-8') as f:
            self.active_hash = json.load(f).get('config_hash', '')
            
        df_audit = pd.read_csv(audit_path)
        if df_audit.empty:
            raise RuntimeError("Audit log is empty. No valid winning model identified in Step 4.")
            
        artifact_hash = df_audit['config_hash'].iloc[0]
        
        if self.active_hash != artifact_hash:
            raise RuntimeError(
                f"\nCRITICAL LINEAGE MISMATCH DETECTED!\n"
                f"Current Configuration Hash: {self.active_hash}\n"
                f"Audit Configuration Hash: {artifact_hash}\n"
                f"Pipeline aborted. Re-execute upstream modeling to align with current configuration."
            )
            
        self.winning_model = df_audit['model_name'].iloc[0]
        nc_file = PATHS['models'] / f"{self.winning_model}_arviz.nc"
        
        if not nc_file.exists():
            raise FileNotFoundError(f"Missing NetCDF artifact for winning model: {nc_file.name}")
            
        self.inference_data = az.from_netcdf(str(nc_file))
        self.logger.info(f"Lineage Verified [{self.active_hash}]. Loaded optimal model: {self.winning_model}")

    def compute_posterior_statistics(self):
        """Calculates exact Bayesian probabilities and HDIs for condition effects."""
        self.logger.info("\nComputing Bayesian hypothesis testing metrics...")
        
        post = self.inference_data.posterior
        variables = list(post.data_vars.keys())
        
        treatment_effects = [v for v in variables if 'C(emotion' in v and 'Treatment' in v]
        
        for param in treatment_effects:
            # Flatten multi-chain posterior array to 1D
            trace_array = post[param].values.flatten()
            
            mean_val = float(np.mean(trace_array))
            median_val = float(np.median(trace_array))
            hdi_bounds = az.hdi(trace_array, hdi_prob=0.95)
            
            # Probability of Direction (Bayesian counterpart to one-sided p-value)
            p_greater_than_zero = float(np.mean(trace_array > 0))
            p_direction = max(p_greater_than_zero, 1.0 - p_greater_than_zero)
            
            # Extract metadata from variable string
            family = param.split('_')[0]
            condition_str = param.split('[T.')[-1].replace(']', '') if '[T.' in param else 'unknown'
            
            self.stats_records.append({
                'Parameter': param,
                'Family': family,
                'Condition': condition_str,
                'Mean': mean_val,
                'Median': median_val,
                'HDI_2.5': float(hdi_bounds[0]),
                'HDI_97.5': float(hdi_bounds[1]),
                'P(>0)': p_greater_than_zero,
                'P(Direction)': p_direction,
                'Significant': int((hdi_bounds[0] > 0) or (hdi_bounds[1] < 0))
            })

        df_stats = pd.DataFrame(self.stats_records)
        df_stats.to_csv(PATHS['tables_main'] / f"bayesian_inference_summary_{self.winning_model}.csv", index=False)
        self.logger.info("Statistical inference table exported successfully.")

    def render_treatment_effects_kde(self):
        """Generates academic overlapping KDE plots representing posterior uncertainties."""
        self.logger.info("Rendering publication-ready posterior density distributions...")
        
        post = self.inference_data.posterior
        families = list(set([r['Family'] for r in self.stats_records]))
        
        for family in families:
            family_records = [r for r in self.stats_records if r['Family'] == family]
            if not family_records: continue
            
            fig, ax = plt.subplots(figsize=(8, 5))
            
            # Formulate the reference zero line for null effects
            ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, zorder=1, label="Null Effect")
            
            y_offset = 0.0  # Vertical spacing for HDI bars to prevent overlap
            
            for record in family_records:
                cond = record['Condition']
                param_name = record['Parameter']
                
                if cond not in CFG.colors: continue
                
                trace_array = post[param_name].values.flatten()
                color = CFG.colors[cond]
                label = CFG.display_labels.get(cond, cond)
                
                # Plot smooth Kernel Density Estimate
                sns.kdeplot(
                    trace_array, fill=True, color=color, alpha=0.5, 
                    linewidth=2, label=label, ax=ax, zorder=3
                )
                
                # Plot 95% HDI as horizontal line at the base
                hdi_low = record['HDI_2.5']
                hdi_high = record['HDI_97.5']
                mean_val = record['Mean']
                
                ax.plot([hdi_low, hdi_high], [y_offset, y_offset], color=color, linewidth=4, solid_capstyle='round', zorder=4)
                ax.scatter([mean_val], [y_offset], color='white', edgecolor=color, zorder=5, s=40, zorder=5)
                
                y_offset -= 0.05 * ax.get_ylim()[1]  # Increment vertical stack down
                
            ax.set_title(f"Posterior Treatment Effects (Δ {CFG.baseline_condition.capitalize()}) - Parameter: {family.upper()}", pad=15, fontweight='bold')
            ax.set_xlabel(f"Estimated Difference in {family.upper()}")
            ax.set_ylabel("Posterior Density")
            
            # Position legend cleanly outside the dense plotting area
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
            
            plt.tight_layout()
            out_pdf = PATHS['figures_main'] / f"FigX_posterior_effects_{family}_{self.winning_model}.pdf"
            plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
            plt.close()

    def run(self):
        """Executes the complete inference and visualization pipeline."""
        if self.inference_data is None: return
        
        try:
            self.compute_posterior_statistics()
            self.render_treatment_effects_kde()
        finally:
            if self.inference_data is not None:
                del self.inference_data
            gc.collect()
            
        self.logger.info("\n" + "=" * 80)
        self.logger.info("FINAL STATISTICAL INFERENCE CONCLUDED.")
        self.logger.info("=" * 80)

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    try:
        engine = BayesianInferenceEngine()
        engine.run()
    except Exception as e:
        print(f"\nCRITICAL PIPELINE FAILURE: {e}")
        import traceback
        traceback.print_exc()

16:19:21 - INFO - ================================================================================
16:19:21 - INFO - POSTERIOR RECONSTRUCTION, CONTRASTS & VISUALIZATION PIPELINE
16:19:21 - INFO - ================================================================================
16:19:21 - INFO - Gatekeeper passed. Target optimal model: [va]
16:19:21 - INFO - Selection Rationale: Rejected: Failed minimum convergence eligibility
16:19:21 - INFO - Loading MCMC posterior arrays from va_emotion_arviz.nc...
16:19:22 - INFO -   Reconstructing marginals for [v]...
16:19:22 - INFO -   Reconstructing marginals for [a]...
16:19:22 - INFO -   Reconstructing marginals for [t]...
16:19:22 - INFO - Exported parameter summary to parameter_summary_va.csv
16:19:22 - INFO - Computing Direct Posterior Contrasts...
16:19:22 - INFO - Exported Direct Contrasts to posterior_direct_contrasts_va.csv
16:19:22 - INFO - Rendering parameter posterior distributions...
16:19:23 - INFO - Rendering Emotion-Specific Rejec

# Step 6: Data Informed Group-Level Parameter Recovery

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: Dual-Track Parameter Recovery Check (Step 6)
Description: 
  - Validates structural identifiability via two parallel forward simulations:
    1. Posterior-Anchored: Samples pseudo-truth from the empirical joint posterior.
    2. Prior-Predictive: Samples pseudo-truth from uniform theoretical bounds.
  - Enforces cryptographic data lineage to ensure configuration consistency.
  - Employs independent multi-chain MCMC with strictly spaced PRNG seeds.
  - Outputs publication-ready diagnostics (Scatter, Bias, Coverage plots) for both modes.
=============================================================================
"""

import os
import gc
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import arviz as az
import hddm
from hddm.generate import gen_rand_data
from scipy.special import expit
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG
except ImportError:
    raise ImportError("CRITICAL ERROR: 'hddm_config.py' not found. Execute Step 2a first.")

PATHS = CFG.initialize_directories()

# Apply standard academic aesthetics (Colorblind friendly, no chart junk)
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'pdf.fonttype': 42, 'ps.fonttype': 42
})

# Define strict physiological and theoretical limits for prior-predictive sampling
THEORETICAL_PRIOR_BOUNDS = {
    'intercepts': {
        'v': [-3.0, 3.0],  # Base drift rate
        'a': [0.6, 2.8],   # Base decision threshold
        't': [0.15, 0.6],  # Base non-decision time (seconds)
        'z': [0.3, 0.7]    # Base starting point ratio
    },
    'effects': {
        'v': [-1.5, 1.5],  # Max plausible shift in drift rate due to emotion
        'a': [-0.5, 0.5],  # Max plausible shift in threshold
        't': [-0.1, 0.1],  # Max plausible shift in NDT
        'z': [-0.1, 0.1]   # Max plausible shift in bias
    }
}

# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_recovery_logger() -> logging.Logger:
    """Initializes standardized logging for the dual-track recovery module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_dual_recovery_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    
    fh = logging.FileHandler(PATHS['audit'] / f'dual_recovery_{ts}.log', encoding='utf-8')
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

# =============================================================================
# CORE CLASS: DUAL-TRACK RECOVERY ENGINE
# =============================================================================
class DualParameterRecoveryEngine:
    """Orchestrates both Posterior-Anchored and Prior-Predictive identifiability checks."""

    def __init__(self, empirical_data_path: str = 'hddm_data_unfair.csv'):
        self.logger = _setup_recovery_logger()
        self.empirical_data_path = empirical_data_path
        
        self.winning_model: str = ""
        self.active_hash: str = ""
        
        # Hyperparameters for Recovery MCMC
        self.recovery_chains = CFG.n_chains if CFG.run_mode != 'debug' else 2
        self.rhat_cut = getattr(CFG, 'rhat_cut', 1.02)
        self.ess_cut = getattr(CFG, 'ess_cut', 400.0)
        self.adaptive_batch_size = getattr(CFG, 'adaptive_batch_size', 2000 if CFG.run_mode != 'debug' else 200)
        self.max_adaptive_rounds = getattr(CFG, 'max_adaptive_rounds', 3 if CFG.run_mode != 'debug' else 1)
        
        self.logger.info("=" * 80)
        self.logger.info(f"DUAL-TRACK PARAMETER RECOVERY INITIATED (Mode: {CFG.run_mode.upper()})")
        
        self._validate_lineage_and_set_winner()

    def _validate_lineage_and_set_winner(self):
        """Cross-validates upstream artifacts to prevent testing against stale states."""
        fingerprint_path = PATHS['manifests'] / "config_fingerprint.json"
        audit_path = PATHS['audit'] / "final_model_selection_audit.csv"
        
        if not fingerprint_path.exists() or not audit_path.exists():
            raise FileNotFoundError("Missing lineage/audit manifests. Run Steps 2a-4 first.")
            
        with open(fingerprint_path, 'r', encoding='utf-8') as f:
            self.active_hash = json.load(f).get('config_hash', '')
            
        df_audit = pd.read_csv(audit_path)
        if df_audit.empty:
            raise RuntimeError("Audit log empty. No valid winning model identified in Step 4.")
            
        artifact_hash = df_audit['config_hash'].iloc[0]
        
        if self.active_hash != artifact_hash:
            raise RuntimeError(
                f"\nCRITICAL LINEAGE MISMATCH DETECTED!\n"
                f"Current Hash: {self.active_hash}\n"
                f"Audit Hash:   {artifact_hash}\n"
                f"Pipeline aborted. Re-execute upstream modeling to align configuration."
            )
            
        self.winning_model = df_audit['model_name'].iloc[0]
        self.logger.info(f"Lineage Verified [{self.active_hash}]. Targeting model: {self.winning_model}")

    def establish_ground_truth(self, mode: str) -> Dict[str, float]:
        """Generates or extracts the pseudo-truth target parameters based on the requested mode."""
        self.logger.info(f"[{mode.upper()}] Establishing Ground Truth Parameters...")
        pseudo_truth_dict = {}
        
        if mode == 'posterior':
            nc_file = PATHS['models'] / f"{self.winning_model}_arviz.nc"
            idata = az.from_netcdf(str(nc_file))
            summary = az.summary(idata, round_to=4)
            
            target_indices = [idx for idx in summary.index if not idx.startswith(('wfpt', 'mc_', '__')) 
                              and ('Intercept' in idx or 'Treatment' in idx)]
            pseudo_truth_dict = summary.loc[target_indices, 'mean'].to_dict()
            del idata, summary
            gc.collect()
            
        elif mode == 'prior':
            # Identify required parameters dynamically from the HDDM structural config
            raw_formulas = CFG.model_specs_templates[self.winning_model]
            
            # Seed PRNG for strict reproducibility of the prior sampling
            np.random.seed(CFG.base_seed + 999)
            
            for f_str in raw_formulas:
                param_family = f_str.split('~')[0].strip()  # 'v', 'a', 't', 'z'
                is_varying = '{ref}' in f_str
                
                # Sample Intercept
                int_low, int_high = THEORETICAL_PRIOR_BOUNDS['intercepts'].get(param_family, [-1, 1])
                pseudo_truth_dict[f"{param_family}_Intercept"] = np.random.uniform(int_low, int_high)
                
                # Sample Condition Effects if varying
                if is_varying:
                    eff_low, eff_high = THEORETICAL_PRIOR_BOUNDS['effects'].get(param_family, [-0.5, 0.5])
                    for emo in CFG.emotion_order:
                        if emo == CFG.baseline_condition:
                            continue
                        effect_key = f'{param_family}_C(emotion, Treatment("{CFG.baseline_condition}"))[T.{emo}]'
                        pseudo_truth_dict[effect_key] = np.random.uniform(eff_low, eff_high)
                        
        # Export for auditing
        pd.Series(pseudo_truth_dict, name='True_Value').to_csv(PATHS['recovery'] / f"ground_truth_dict_mode_{mode}.csv")
        return pseudo_truth_dict

    def _apply_link_and_clip(self, param: str, lp_val: float) -> float:
        """Applies requisite link functions and imposes physical bounds for data generation."""
        if param == 'v':
            return float(np.clip(lp_val, -8.0, 8.0))
        elif param in ['a', 't']:
            val = np.exp(lp_val)
            if param == 'a': return float(np.clip(val, 0.2, 4.0))
            if param == 't': return float(np.clip(val, 0.05, 2.0))
        elif param == 'z':
            return float(np.clip(expit(lp_val), 0.05, 0.95))
        return lp_val

    def generate_trialwise_skeleton(self, mode: str, truth_dict: Dict[str, float]) -> str:
        """Builds explicit synthetic datasets maintaining the original trial structure."""
        self.logger.info(f"[{mode.upper()}] Executing Trial-Skeleton Driven Forward Simulation...")
        emp_df = pd.read_csv(self.empirical_data_path)
        
        condition_log = []
        trial_rows = []
        
        # Distinct seed per mode to avoid correlated datasets
        seed_offset = 777 if mode == 'posterior' else 888
        np.random.seed(CFG.base_seed + seed_offset)
        
        for (subj, emo), group in emp_df.groupby(['subj_idx', 'emotion']):
            n_trials = len(group)
            phys_params = {}
            
            for p in ['v', 'a', 't', 'z']:
                lp_val = truth_dict.get(f"{p}_Intercept", 0.0)
                if emo != CFG.baseline_condition:
                    effect_key = f'{p}_C(emotion, Treatment("{CFG.baseline_condition}"))[T.{emo}]'
                    lp_val += truth_dict.get(effect_key, 0.0)
                phys_params[p] = self._apply_link_and_clip(p, lp_val)
            
            cond_entry = {'subj_idx': subj, 'emotion': emo, 'n_trials': n_trials}
            cond_entry.update({f"{k}_true": v for k, v in phys_params.items()})
            condition_log.append(cond_entry)
            
            sim_res = gen_rand_data(phys_params, size=n_trials)
            sim_df = sim_res[0] if isinstance(sim_res, tuple) else sim_res
            
            group_copy = group.copy().reset_index(drop=True)
            group_copy['rt_sim'] = sim_df['rt'].values
            group_copy['response_sim'] = sim_df['response'].values
            for p in ['v', 'a', 't', 'z']:
                group_copy[f'{p}_true'] = phys_params[p]
                
            trial_rows.append(group_copy)
            
        final_cond_df = pd.DataFrame(condition_log)
        final_cond_df.to_csv(PATHS['recovery'] / f"recovery_conditionwise_true_params_mode_{mode}.csv", index=False)
        
        final_synth_df = pd.concat(trial_rows, ignore_index=True)
        final_synth_df = final_synth_df.rename(columns={'rt': 'rt_emp', 'response': 'response_emp', 
                                                        'rt_sim': 'rt', 'response_sim': 'response'})
        
        synth_path = PATHS['recovery'] / f"recovery_trialwise_simulation_table_mode_{mode}.csv"
        final_synth_df.to_csv(synth_path, index=False)
        self.logger.info(f"[{mode.upper()}] Synthetic cohort generated: {len(final_synth_df)} trials mapped.")
        
        return str(synth_path)

    def _evaluate_recovery_convergence(self, models: List[hddm.HDDMRegressor]) -> Tuple[bool, Dict[str, float]]:
        """Evaluates convergence explicitly for the recovery chains."""
        traces = [m.get_traces() for m in models]
        valid_cols = [c for c in traces[0].columns if not c.startswith(('wfpt', 'mc_', '__'))]
        
        post_dict = {col: np.stack([t[col].values for t in traces]) for col in valid_cols}
        idata = az.from_dict(posterior=post_dict)
        summary = az.summary(idata, round_to=4)
        
        max_rhat = float(summary['r_hat'].max())
        min_ess = float(summary['ess_bulk'].min())
        
        is_converged = (max_rhat <= self.rhat_cut) and (min_ess >= self.ess_cut)
        metrics = {'max_rhat': max_rhat, 'min_ess': min_ess}
        
        del traces, post_dict, idata, summary
        gc.collect()
        return is_converged, metrics

    def execute_independent_multichain_refit(self, mode: str, synth_path: str):
        """Performs multi-chain estimation ensuring structurally independent Markov chains."""
        self.logger.info(f"\n[{mode.upper()}] Re-estimating [{self.winning_model}] via Independent Multi-Chain Protocol...")
        df_synth = pd.read_csv(synth_path)
        df_synth['subj_idx'] = df_synth['subj_idx'].astype(str)
        
        ref_str = f'C(emotion, Treatment("{CFG.baseline_condition}"))'
        formatted_formulas = [f.format(ref=ref_str) for f in CFG.model_specs_templates[self.winning_model]]
        
        base_args = {
            'include': ['v', 'a', 't', 'z'], 'is_group_model': True,
            'group_only_regressors': getattr(CFG, 'group_only_regressors', True),
            'keep_regressor_trace': getattr(CFG, 'keep_regressor_trace', False),
            'p_outlier': CFG.p_outlier, 'informative': CFG.use_informative_priors
        }
        
        models = []
        for chain_idx in range(self.recovery_chains):
            # STRICT SEED SPACING: prevent PRNG overlap.
            chain_seed = CFG.base_seed + (chain_idx * 1000) + (10000 if mode == 'prior' else 0)
            np.random.seed(chain_seed)
            self.logger.info(f"  -> Initializing Chain {chain_idx + 1}/{self.recovery_chains} (Seed: {chain_seed})")
            
            model_c = hddm.HDDMRegressor(df_synth, formatted_formulas, **base_args)
            try: model_c.find_starting_values()
            except Exception: self.logger.warning(f"     [Warning] MAP optimization failed for Chain {chain_idx}.")
            
            db_path = str(PATHS['recovery'] / f"recovery_{self.winning_model}_chain_{chain_idx}_mode_{mode}.db")
            model_c.sample(CFG.n_samples, burn=CFG.n_burn, thin=CFG.thin, dbname=db_path, db='pickle')
            models.append(model_c)

        # Adaptive Continuation Loop
        current_samples = CFG.n_samples
        rounds = 0
        final_metrics = {'max_rhat': np.nan, 'min_ess': np.nan}
        
        if self.recovery_chains > 1:
            while rounds < self.max_adaptive_rounds:
                is_converged, metrics = self._evaluate_recovery_convergence(models)
                final_metrics = metrics
                if is_converged:
                    self.logger.info(f"     [Recovery Convergence] Achieved at Round {rounds}. (R-hat: {metrics['max_rhat']:.3f})")
                    break
                self.logger.info(f"     [Adaptive] Suboptimal metrics (R-hat: {metrics['max_rhat']:.3f}). Appending +{self.adaptive_batch_size} sweeps...")
                for chain_idx, model_c in enumerate(models):
                    db_path = str(PATHS['recovery'] / f"recovery_{self.winning_model}_chain_{chain_idx}_mode_{mode}.db")
                    model_c.sample(self.adaptive_batch_size, burn=0, thin=CFG.thin, dbname=db_path, db='pickle')
                current_samples += self.adaptive_batch_size
                rounds += 1
                
        # Final Summary Extraction
        self.logger.info(f"[{mode.upper()}] Extracting final recovered summary...")
        traces = [m.get_traces() for m in models]
        valid_cols = [c for c in traces[0].columns if not c.startswith(('wfpt', 'mc_', '__'))]
        post_dict = {col: np.stack([t[col].values for t in traces]) for col in valid_cols}
        idata = az.from_dict(posterior=post_dict)
        rec_summary = az.summary(idata, round_to=4)
        rec_summary.to_csv(PATHS['recovery'] / f"recovery_fit_summary_{self.winning_model}_mode_{mode}.csv")
        
        for m in models:
            if hasattr(m, 'db'):
                try: m.db.close()
                except: pass
        del models, traces, post_dict, idata
        gc.collect()

    def calculate_multidimensional_metrics(self, mode: str, truth_dict: Dict[str, float]) -> pd.DataFrame:
        """Computes granular recovery metrics mapping posterior parameters against the pseudo-truth."""
        self.logger.info(f"\n[{mode.upper()}] Calculating structured recovery metrics...")
        rec_summary = pd.read_csv(PATHS['recovery'] / f"recovery_fit_summary_{self.winning_model}_mode_{mode}.csv", index_col=0)
        
        records = []
        for param, gt_val in truth_dict.items():
            if param not in rec_summary.index: continue
            
            rec_row = rec_summary.loc[param]
            rec_mean = rec_row['mean']
            hdi_low = rec_row['hdi_3%']
            hdi_high = rec_row['hdi_97%']
            
            bias = rec_mean - gt_val
            rel_bias = bias / np.abs(gt_val) if np.abs(gt_val) > 1e-4 else np.nan
            is_covered = hdi_low <= gt_val <= hdi_high
            
            family = param.split('_')[0] if '_' in param else 'other'
            param_type = 'Intercept' if 'Intercept' in param else 'ConditionEffect'
            
            records.append({
                'Parameter': param, 'Family': family, 'Type': param_type,
                'Ground_Truth': gt_val, 'Recovered_Mean': rec_mean,
                'Bias': bias, 'Abs_Bias': np.abs(bias), 'Relative_Bias': rel_bias,
                'Sq_Error': bias**2, 'HDI_3': hdi_low, 'HDI_97': hdi_high,
                'HDI_Width': hdi_high - hdi_low, 'Coverage': int(is_covered)
            })

        df_metrics = pd.DataFrame(records)
        df_metrics.to_csv(PATHS['recovery'] / f"recovery_detailed_metrics_mode_{mode}.csv", index=False)
        
        summary = df_metrics.groupby(['Family', 'Type']).agg(
            N=('Parameter', 'count'),
            RMSE=('Sq_Error', lambda x: np.sqrt(x.mean())),
            Mean_HDI_Width=('HDI_Width', 'mean'),
            Coverage_Rate=('Coverage', 'mean')
        ).reset_index()
        self.logger.info(f"\n{summary.to_string(index=False)}")
        return df_metrics

    def render_publication_diagnostics(self, mode: str, df_plot: pd.DataFrame):
        """Renders comprehensive academic visual diagnostics for parameter recovery."""
        self.logger.info(f"[{mode.upper()}] Rendering publication-grade visual diagnostics...")
        if df_plot.empty: return
        
        pal = {'v': '#3C5488', 'a': '#E64B35', 't': '#00A087', 'z': '#8491B4'}
        
        # Plot 1: Standard Scatter
        g = sns.FacetGrid(df_plot, col="Family", col_wrap=2, sharex=False, sharey=False, height=3.5)
        def custom_scatter(x, y, **kwargs):
            ax = plt.gca()
            ax.scatter(x, y, alpha=0.8, edgecolors='w', s=60, **kwargs)
            lims = [np.min([ax.get_xlim(), ax.get_ylim()]), np.max([ax.get_xlim(), ax.get_ylim()])]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
        
        g.map_dataframe(custom_scatter, x="Ground_Truth", y="Recovered_Mean", hue="Family", palette=pal)
        g.set_axis_labels("Pseudo-Truth", "Recovered Posterior Mean")
        g.set_titles(col_template="{col_name} Parameter Space")
        title_prefix = "Posterior-Anchored" if mode == 'posterior' else "Prior-Predictive"
        plt.suptitle(f"{title_prefix} Recovery Scatter ({self.winning_model.upper()})", y=1.05, fontweight='bold')
        plt.savefig(PATHS['figures_supp'] / f"FigS6a_recovery_scatter_{self.winning_model}_mode_{mode}.pdf", dpi=300, bbox_inches='tight')
        plt.close()

        # Plot 2: Interval Coverage Plot
        df_plot_sorted = df_plot.sort_values(by=['Family', 'Type'])
        fig, ax = plt.subplots(figsize=(8, len(df_plot_sorted) * 0.4))
        for i, (_, row) in enumerate(df_plot_sorted.iterrows()):
            color = pal.get(row['Family'], 'black')
            ax.plot([row['HDI_3'], row['HDI_97']], [i, i], color=color, linewidth=3, alpha=0.5)
            ax.scatter(row['Recovered_Mean'], i, color=color, s=60, zorder=3)
            ax.scatter(row['Ground_Truth'], i, color='red', marker='x', s=100, linewidths=2, zorder=4)
            
        ax.set_yticks(range(len(df_plot_sorted)))
        ax.set_yticklabels(df_plot_sorted['Parameter'])
        ax.set_title(f"{title_prefix}: 95% HDI Interval Coverage", pad=15, fontweight='bold')
        ax.set_xlabel("Parameter Posterior Space")
        
        red_cross = mlines.Line2D([], [], color='red', marker='x', linestyle='None', markersize=8, label='Pseudo-Truth')
        blue_dot = mlines.Line2D([], [], color='gray', marker='o', linestyle='None', markersize=6, label='Recovered Mean & 95% HDI')
        ax.legend(handles=[red_cross, blue_dot], bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        plt.savefig(PATHS['figures_supp'] / f"FigS6b_recovery_coverage_{self.winning_model}_mode_{mode}.pdf", dpi=300, bbox_inches='tight')
        plt.close()

    def run_full_pipeline(self):
        """Orchestrates the sequential execution of both recovery modes."""
        for mode in ['posterior', 'prior']:
            self.logger.info("\n" + "*" * 60)
            self.logger.info(f"COMMENCING MODULE: {mode.upper()} RECOVERY")
            self.logger.info("*" * 60)
            
            truth_dict = self.establish_ground_truth(mode)
            synth_path = self.generate_trialwise_skeleton(mode, truth_dict)
            self.execute_independent_multichain_refit(mode, synth_path)
            metrics_df = self.calculate_multidimensional_metrics(mode, truth_dict)
            self.render_publication_diagnostics(mode, metrics_df)
            
        self.logger.info("\n" + "=" * 80)
        self.logger.info("DUAL-TRACK PARAMETER RECOVERY COMPLETED SUCCESSFULLY")
        self.logger.info("=" * 80)

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    try:
        engine = DualParameterRecoveryEngine()
        engine.run_full_pipeline()
    except Exception as e:
        print(f"\nCRITICAL PIPELINE FAILURE: {e}")
        import traceback
        traceback.print_exc()